# Benchmark Wan 2.2 TI2V 5B trên Kaggle hoặc Colab

Notebook này chỉ tạo mẫu video để đánh giá G1. ComfyUI chạy nội bộ trong phiên notebook trên địa chỉ loopback; notebook không mở tunnel, không phục vụ API ra Internet và không dùng dữ liệu thật của dự án.

Quy trình tạo tuần tự ba mẫu T2V dọc, một mẫu I2V từ khung tĩnh và một mẫu nối tiếp từ khung cuối của clip trước. Mỗi clip dùng 49 khung ở 16 fps, sau đó được chuẩn hóa thành MP4 dọc 1080 x 1920, 30 fps. Tổng tải model khoảng 16.9 GB; cần ít nhất 22 GiB trống ở vùng scratch.

Trước khi chạy, đọc điều khoản của nền tảng đã chọn. Google Colab managed chỉ dùng cho tính toán tương tác trong notebook; không mở web service/tunnel. Kaggle chỉ được dùng khi điều khoản và quyền sử dụng cho dự án này phù hợp. Chưa tự bật hoặc chấp nhận điều khoản thay người dùng.

Nguồn workflow ComfyUI và model được khóa revision, hash và ghi vào manifest khi chạy. Wan 2.2 TI2V 5B theo giấy phép Apache 2.0. Output và manifest được lưu trong thư mục kết quả của phiên để xem/tải xuống.

## 1. Cấu hình và xác nhận trước khi tải

Đặt các giá trị xác nhận ở ô kế tiếp sau khi tự chọn nền tảng, xem điều khoản và sẵn sàng tải khoảng 16.9 GB model. Giá trị mặc định dừng an toàn; chạy cell tiếp theo sẽ không tải model nếu chưa bật các xác nhận.

In [ ]:
# Chỉ đổi các giá trị này sau khi bạn đã tự kiểm tra nền tảng và sẵn sàng tải model.
DIEU_KHOAN_DA_DOC = False
KAGGLE_DUOC_PHEP_CHO_DU_AN = False
BAT_DAU_TAI_MODEL = False

# Hồ sơ G1 đã duyệt trong plan.
PROFILE = {"width": 480, "height": 832, "frames": 49, "fps": 16, "steps": 24, "cfg": 5.0}
T2V_PROMPTS = [
    "Một trợ lý nhỏ bằng giấy gấp origami ngồi trên bàn làm việc, lắng nghe rồi gật đầu nhẹ; máy quay tiến chậm, ánh sáng ấm, chuyển động tự nhiên, không có chữ đọc được.",
    "Một tấm thẻ công việc bằng giấy trượt qua ba bàn xử lý thủ công; mỗi lần chuyển có một bàn tay đặt thẻ xuống gọn gàng, máy quay lia ngang chậm, không có chữ đọc được.",
    "Một người vận hành xem lại kết quả trên tấm thẻ giấy, dừng lại suy nghĩ rồi đánh dấu bằng bút chì; cận cảnh bàn tay, ánh sáng dịu, không có chữ đọc được."
]
I2V_PROMPT = "Trợ lý giấy gấp origami bắt đầu chuyển động, nghiêng đầu và đưa tay chỉ về phía trước; giữ nguyên kiểu giấy, màu sắc và bố cục của khung hình đầu."
NOI_DUNG_NOI_TIEP = "Cùng trợ lý origami tiếp tục chuyển động từ đúng tư thế hiện tại, đưa tấm thẻ công việc tới bàn rà soát; máy quay tiến nhẹ, giữ nguyên màu sắc và bố cục."
NEGATIVE_PROMPT = "chữ, phụ đề, logo, watermark, hình đứng yên, nhòe, méo hình, thêm ngón tay, vật thể biến dạng, khung hình giật"


In [ ]:
import os, sys, json, time, shutil, socket, subprocess, hashlib, urllib.request
from pathlib import Path

IS_KAGGLE = Path("/kaggle").exists()
IS_COLAB = Path("/content").exists() and not IS_KAGGLE
if not (IS_KAGGLE or IS_COLAB):
    raise RuntimeError("Notebook chỉ hỗ trợ Kaggle hoặc Colab managed runtime.")
if not DIEU_KHOAN_DA_DOC:
    raise RuntimeError("Hãy đọc điều khoản nền tảng, rồi bật DIEU_KHOAN_DA_DOC.")
if IS_KAGGLE and not KAGGLE_DUOC_PHEP_CHO_DU_AN:
    raise RuntimeError("Chỉ chạy trên Kaggle khi bạn đã xác nhận điều khoản cho phép dùng trong dự án này.")
if IS_COLAB:
    print("Colab: chỉ chạy tính toán tương tác trong notebook; không mở tunnel hoặc API công khai.")
if not BAT_DAU_TAI_MODEL:
    raise RuntimeError("Hãy bật BAT_DAU_TAI_MODEL khi đã sẵn sàng tải khoảng 16.9 GB.")

if IS_KAGGLE:
    WORK_ROOT, PERSIST_ROOT = Path("/kaggle/temp/wan_benchmark"), Path("/kaggle/working/wan_benchmark")
else:
    WORK_ROOT = PERSIST_ROOT = Path("/content/wan_benchmark")
WORK_ROOT.mkdir(parents=True, exist_ok=True)
ARTIFACTS = WORK_ROOT / "ket_qua"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
scratch = shutil.disk_usage(WORK_ROOT)
ram_gib = round(os.sysconf("SC_PAGE_SIZE")*os.sysconf("SC_PHYS_PAGES")/1024**3,2)
print({"nen_tang":"Kaggle" if IS_KAGGLE else "Colab",
       "scratch_con_gib":round(scratch.free/1024**3,2),"ram_gib":ram_gib})
if scratch.free < 22*1024**3:
    raise RuntimeError("Scratch còn dưới 22 GiB; dừng trước khi tải model.")


## 2. Kiểm tra GPU, RAM, Internet và outbound tunnel

Cell này chỉ ghi nhận khả năng mạng; không khởi tạo tunnel. Nếu cổng 7844 bị chặn, benchmark nội bộ vẫn có thể chạy, nhưng tunnel sau này sẽ cần host/mạng khác.

In [ ]:
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                       capture_output=True, text=True)
if gpu.returncode:
    raise RuntimeError("Không tìm thấy GPU NVIDIA: " + gpu.stderr[-500:])
gpus = [line.strip() for line in gpu.stdout.splitlines() if line.strip()]
print("GPU được cấp:", gpus)
if len(gpus) > 1:
    print("Chỉ dùng GPU 0; không ghép nhiều GPU trong benchmark này.")
mem_mib = int(gpus[0].split(",")[-1].strip().split()[0])
if mem_mib < 8192:
    raise RuntimeError("GPU có dưới 8 GiB VRAM; dừng trước khi tải model.")
try:
    with urllib.request.urlopen("https://huggingface.co/api/models/Comfy-Org/Wan_2.2_ComfyUI_Repackaged", timeout=20) as r:
        print("Internet HTTPS tới Hugging Face:", r.status)
except Exception as e:
    raise RuntimeError("Không truy cập được nguồn model: " + str(e))
try:
    with socket.create_connection(("region1.v2.argotunnel.com", 7844), timeout=5):
        tunnel_7844 = "cho_phep"
except Exception as e:
    tunnel_7844 = "bi_chan: " + type(e).__name__
print("Outbound TCP/7844:", tunnel_7844)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"


## 3. Cài ComfyUI và tải đúng ba tệp model

ComfyUI và workflow được khóa theo commit chính thức. Model revisions được chốt ngay trước khi tải; checksum SHA-256 thực tế được lưu. Không cần Hugging Face token vì các tệp nguồn là công khai.

In [ ]:
COMFY_COMMIT = "b5a432157de6b8884f81104cd02926adf4d597e4"
WORKFLOW_COMMIT = "abb9cf6477b59c52dbe234c67f82318dadd872b2"
WORKFLOW_SHA256 = "23C93B88ADECBDF5BCF4C5438117070FE757F30ACA373DD6A2EB488D2C6A585A"
COMFY = WORK_ROOT / "ComfyUI"
if not COMFY.exists():
    subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout",
                    "https://github.com/Comfy-Org/ComfyUI.git", str(COMFY)], check=True)
subprocess.run(["git", "-C", str(COMFY), "checkout", "--detach", COMFY_COMMIT], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(COMFY/"requirements.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "huggingface_hub", "requests"], check=True)

from huggingface_hub import hf_hub_download
import requests
MODEL_FILES = [
    ("Comfy-Org/Wan_2.2_ComfyUI_Repackaged", "split_files/diffusion_models/wan2.2_ti2v_5B_fp16.safetensors", COMFY/"models/diffusion_models"),
    ("Comfy-Org/Wan_2.2_ComfyUI_Repackaged", "split_files/vae/wan2.2_vae.safetensors", COMFY/"models/vae"),
    ("Comfy-Org/Wan_2.1_ComfyUI_repackaged", "split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors", COMFY/"models/text_encoders"),
]
def sha256_file(path):
    digest=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(8*1024*1024),b""):
            digest.update(chunk)
    return digest.hexdigest()
model_locks, download_seconds = {}, {}
for repo, filename, dest in MODEL_FILES:
    info=requests.get("https://huggingface.co/api/models/"+repo,timeout=30)
    info.raise_for_status()
    revision=info.json()["sha"]
    dest.mkdir(parents=True,exist_ok=True)
    start=time.monotonic()
    path=Path(hf_hub_download(repo_id=repo,filename=filename,revision=revision,
                              local_dir=dest,cache_dir=WORK_ROOT/"hf_cache"))
    download_seconds[filename]=round(time.monotonic()-start,2)
    model_locks[repo+"/"+filename]={"revision":revision,"sha256":sha256_file(path),
        "bytes":path.stat().st_size,"download_seconds":download_seconds[filename]}
print(json.dumps(model_locks,ensure_ascii=False,indent=2))


## 4. Mở ComfyUI nội bộ

ComfyUI chỉ bind vào 127.0.0.1. Không có lệnh cloudflared, ngrok, SSH tunnel hoặc cổng public trong notebook.

In [ ]:
import requests
COMFY_URL="http://127.0.0.1:8188"
server_log=open(WORK_ROOT/"comfyui.log","w",encoding="utf-8")
server=subprocess.Popen([sys.executable,"main.py","--listen","127.0.0.1","--port","8188"],
                        cwd=COMFY,stdout=server_log,stderr=subprocess.STDOUT)
server_start=time.monotonic()
for _ in range(120):
    if server.poll() is not None:
        raise RuntimeError("ComfyUI dừng khi khởi động; xem comfyui.log.")
    try:
        r=requests.get(COMFY_URL+"/system_stats",timeout=2)
        if r.ok: break
    except requests.RequestException: pass
    time.sleep(2)
else: raise TimeoutError("ComfyUI chưa sẵn sàng sau 4 phút.")
server_ready_seconds=round(time.monotonic()-server_start,2)
object_info=requests.get(COMFY_URL+"/object_info",timeout=30).json()
print("ComfyUI sẵn sàng trong giây:",server_ready_seconds,"Số loại node:",len(object_info))


## 5. Khóa workflow và chuẩn bị yêu cầu

Workflow tải từ commit đã pin và phải khớp SHA-256. Các prompt mẫu bám storyboard origami đã duyệt ở G0. Mỗi lần tạo video được gửi riêng lẻ và không tự gửi lại nếu kết quả POST bị mơ hồ.

In [ ]:
import copy
WORKFLOW_URL=(f"https://raw.githubusercontent.com/Comfy-Org/workflow_templates/{WORKFLOW_COMMIT}"
              "/templates/video_wan2_2_5B_ti2v.json")
raw=requests.get(WORKFLOW_URL,timeout=30)
raw.raise_for_status()
if hashlib.sha256(raw.content).hexdigest().upper()!=WORKFLOW_SHA256:
    raise RuntimeError("Hash workflow không khớp; dừng để rà soát nguồn.")
template=raw.json()
template["nodes"]=[n for n in template["nodes"] if n.get("type")!="MarkdownNote"]
state_path=WORK_ROOT/"trang_thai_run.json"
state=json.loads(state_path.read_text(encoding="utf-8")) if state_path.exists() else {}

def save_state():
    state_path.write_text(json.dumps(state,ensure_ascii=False,indent=2),encoding="utf-8")

def workflow_to_api(wf):
    links={x[0]:x[1:] for x in wf.get("links",[])}
    active={n["id"] for n in wf["nodes"] if n.get("mode",0) not in (2,4)}
    result={}
    for n in wf["nodes"]:
        if n["id"] not in active: continue
        typ=n["type"]; info=object_info.get(typ)
        if not info: raise RuntimeError("ComfyUI thiếu node: "+typ)
        inputs={}; sockets={i["name"] for i in n.get("inputs",[])}
        definitions=info.get("input",{})
        names=[k for section in ("required","optional") for k in definitions.get(section,{}) if k not in sockets]
        values=list(n.get("widgets_values",[]))
        if typ=="KSampler" and len(values)==7: values.pop(1)  # control_after_generate ch? l? widget giao di?n.
        if typ=="LoadImage": names,values=["image"],values[:1]
        if len(names)!=len(values):
            raise RuntimeError(f"Không ánh xạ được widget node {typ}: {names} / {values}")
        inputs.update(zip(names,values))
        for inp in n.get("inputs",[]):
            lid=inp.get("link")
            if lid is None or lid not in links: continue
            origin,slot,target,_,_=links[lid]
            if origin in active and target==n["id"]:
                inputs[inp["name"]]=[str(origin),int(slot)]
        result[str(n["id"])]={"class_type":typ,"inputs":inputs,
                              "_meta":{"title":n.get("title",typ)}}
    return result

def widget(wf,node_id,index,value):
    next(n for n in wf["nodes"] if n["id"]==node_id)["widgets_values"][index]=value

def prepare_workflow(name,prompt,seed,image_name=None):
    wf=copy.deepcopy(template)
    widget(wf,55,0,PROFILE["width"]); widget(wf,55,1,PROFILE["height"])
    widget(wf,55,2,PROFILE["frames"]); widget(wf,57,0,PROFILE["fps"])
    widget(wf,58,0,"video/wan22_"+name); widget(wf,6,0,prompt)
    widget(wf,7,0,NEGATIVE_PROMPT); widget(wf,3,0,int(seed))
    widget(wf,3,1,"fixed"); widget(wf,3,2,PROFILE["steps"]); widget(wf,3,3,PROFILE["cfg"])
    image=next(n for n in wf["nodes"] if n["id"]==56)
    if image_name:
        image["mode"]=0; widget(wf,56,0,image_name)
    else: image["mode"]=4
    return workflow_to_api(wf)

# T? ki?m tra chuy?n workflow, kh?ng g?i y?u c?u inference.
probe_t2v=prepare_workflow("kiem_tra_t2v",T2V_PROMPTS[0],654321)
probe_i2v=prepare_workflow("kiem_tra_i2v",I2V_PROMPT,654322,"khung_tinh.png")
assert probe_t2v["55"]["inputs"]["width"]==PROFILE["width"]
assert probe_t2v["55"]["inputs"]["length"]==PROFILE["frames"]
assert "56" not in probe_t2v and "start_image" not in probe_t2v["55"]["inputs"]
assert probe_i2v["55"]["inputs"]["start_image"]==["56",0]
assert probe_i2v["56"]["inputs"]["image"]=="khung_tinh.png"
assert probe_t2v["3"]["inputs"]["steps"]==PROFILE["steps"]
assert probe_t2v["3"]["inputs"]["seed"]==654321
assert sum(x["class_type"]=="SaveVideo" for x in probe_i2v.values())==1
print("T? ki?m tra chuy?n workflow ??t.")


## 6. Chạy tuần tự T2V, I2V và nối clip

Cell này không chạy song song. Nếu một yêu cầu đang chạy, timeout hoặc cho kết quả chưa rõ, trạng thái được giữ lại để tránh gửi inference trùng. Hãy đối soát queue/history trước khi quyết định thao tác tiếp.

In [ ]:
def submit_video(name,prompt,seed,image_name=None):
    old=state.get(name)
    if old:
        if old.get("status")=="completed" and Path(old.get("path","")).exists():
            return Path(old["path"]),old
        raise RuntimeError("Run đã tồn tại ở trạng thái "+old.get("status","unknown")+
                           "; không tự gửi lại. Đối soát queue/log trước.")
    graph=prepare_workflow(name,prompt,seed,image_name)
    state[name]={"status":"submitting","prompt_sha256":hashlib.sha256(prompt.encode()).hexdigest(),
                 "seed":seed,"started_at":time.time()}
    save_state()
    try:
        response=requests.post(COMFY_URL+"/prompt",json={"prompt":graph},timeout=(5,60))
        response.raise_for_status(); prompt_id=response.json()["prompt_id"]
    except Exception:
        state[name]["status"]="unknown"; save_state(); raise
    state[name].update(status="queued",prompt_id=prompt_id); save_state()
    deadline,peak_vram=time.monotonic()+7200,0
    while time.monotonic()<deadline:
        history=requests.get(COMFY_URL+"/history/"+prompt_id,timeout=20).json()
        entry=history.get(prompt_id)
        if entry and entry.get("outputs"):
            videos=entry["outputs"].get("58",{}).get("videos",[])
            if not videos:
                state[name]["status"]="failed"; save_state()
                raise RuntimeError("Run xong nhưng node SaveVideo không trả video.")
            item=videos[0]
            source=(COMFY/"output"/item.get("subfolder","")/item["filename"]).resolve()
            if COMFY/"output" not in source.parents:
                raise RuntimeError("Đường dẫn output nằm ngoài thư mục ComfyUI/output.")
            state[name].update(status="completed",path=str(source),
                seconds=round(time.monotonic()-state[name]["started_at"],2),
                peak_vram_mib=peak_vram,output_bytes=source.stat().st_size)
            save_state(); return source,state[name]
        if entry and entry.get("status",{}).get("completed"):
            state[name]["status"]="failed"; save_state()
            raise RuntimeError("ComfyUI ho?n t?t job nh?ng kh?ng c? output; ki?m tra log tr??c khi ch?y l?i.")
        if server.poll() is not None:
            state[name]["status"]="unknown"; save_state()
            raise RuntimeError("ComfyUI dừng giữa inference; không gửi lại tự động.")
        mem=subprocess.run(["nvidia-smi","--query-gpu=memory.used","--format=csv,noheader,nounits"],
                           capture_output=True,text=True)
        if mem.returncode==0:
            peak_vram=max(peak_vram,max([int(x) for x in mem.stdout.splitlines() if x.strip()] or [0]))
        time.sleep(3)
    state[name]["status"]="unknown"; save_state()
    raise TimeoutError("Quá hạn chờ; giữ trạng thái để không tạo inference trùng.")


## 7. Chạy bộ G1 và chuẩn hóa file

Ba prompt T2V chạy trước. Từ clip đầu, notebook lấy khung đầu làm ảnh tĩnh I2V và khung cuối làm điểm nối cho clip tiếp theo. Việc chọn/crop chủ thể vẫn cần người xem kiểm tra bằng video.

In [ ]:
import math
from fractions import Fraction

def sha256_path(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(8*1024*1024),b""): h.update(chunk)
    return h.hexdigest()

def extract_frame(video,output,last=False):
    seek=["-sseof","-0.08"] if last else []
    subprocess.run(["ffmpeg","-hide_banner","-loglevel","error","-y",*seek,"-i",str(video),
                    "-frames:v","1",str(output)],check=True)
    return Path(output)

def probe(video):
    data=json.loads(subprocess.check_output(["ffprobe","-v","error","-show_streams",
                      "-show_format","-of","json",str(video)],text=True))
    v=next(x for x in data["streams"] if x["codec_type"]=="video")
    return {"width":v["width"],"height":v["height"],"fps":float(Fraction(v["avg_frame_rate"])),
            "duration_seconds":float(data["format"]["duration"]),
            "codec":v["codec_name"],"bytes":Path(video).stat().st_size}

def normalize(video,name):
    out=ARTIFACTS/(name+"_1080x1920.mp4")
    subprocess.run(["ffmpeg","-hide_banner","-loglevel","error","-y","-i",str(video),
        "-vf","scale=1080:1920:force_original_aspect_ratio=increase,crop=1080:1920,fps=30",
        "-an","-c:v","libx264","-pix_fmt","yuv420p","-movflags","+faststart",str(out)],check=True)
    return out

results=[]
started=time.monotonic()
for i,prompt in enumerate(T2V_PROMPTS,1):
    src,meta=submit_video(f"t2v_{i:02d}",prompt,4100+i)
    meta["inference_phase"]="cold_first_includes_model_load" if i==1 else "warm"
    raw_out=ARTIFACTS/f"t2v_{i:02d}_goc{src.suffix}"; shutil.copy2(src,raw_out)
    norm=normalize(raw_out,f"t2v_{i:02d}")
    results.append({"name":f"t2v_{i:02d}","raw":str(raw_out),"normalized":str(norm),
                    "metrics":meta,"probe_raw":probe(raw_out),"probe_normalized":probe(norm)})

first=Path(results[0]["raw"])
still=extract_frame(first,WORK_ROOT/"khung_tinh.png")
shutil.copy2(still,COMFY/"input"/"khung_tinh.png")
src,meta=submit_video("i2v_tu_khung_tinh",I2V_PROMPT,4201,"khung_tinh.png")
raw_out=ARTIFACTS/f"i2v_tu_khung_tinh_goc{src.suffix}"; shutil.copy2(src,raw_out)
norm=normalize(raw_out,"i2v_tu_khung_tinh")
results.append({"name":"i2v_tu_khung_tinh","raw":str(raw_out),"normalized":str(norm),
                "metrics":meta,"probe_raw":probe(raw_out),"probe_normalized":probe(norm)})

last=extract_frame(first,WORK_ROOT/"khung_noi_tiep.png",last=True)
src,meta=submit_video("i2v_noi_tiep",NOI_DUNG_NOI_TIEP,4202,last.name)
raw_out=ARTIFACTS/f"i2v_noi_tiep_goc{src.suffix}"; shutil.copy2(src,raw_out)
norm=normalize(raw_out,"i2v_noi_tiep")
results.append({"name":"i2v_noi_tiep","raw":str(raw_out),"normalized":str(norm),
                "metrics":meta,"probe_raw":probe(raw_out),"probe_normalized":probe(norm),
                "frame_vao_sha256":sha256_path(last),"frame_vao_bytes":last.stat().st_size})
print("Thời gian bộ render, giây:",round(time.monotonic()-started,2))


## 8. Lưu bằng chứng, xem và tải video

Kết quả gồm video gốc, bản chuẩn hóa, hash model/workflow, số đo runtime và thời gian tạo. Các video chưa tự động được duyệt về bố cục; hãy mở và xem toàn clip trước khi chốt profile.

In [ ]:
manifest={
 "nen_tang":"Kaggle" if IS_KAGGLE else "Colab","benchmark_only":True,"tunnel_opened":False,
 "comfyui_commit":COMFY_COMMIT,"workflow_commit":WORKFLOW_COMMIT,
 "workflow_sha256":WORKFLOW_SHA256,"workflow_url":WORKFLOW_URL,"profile":PROFILE,
 "gpu":gpus,"ram_gib":ram_gib,"outbound_tcp_7844":tunnel_7844,
 "comfyui_start_seconds":server_ready_seconds,"download_seconds":download_seconds,
 "model_locks":model_locks,"results":results,"run_state":state}
for item in results:
    for key in ("raw","normalized"): item[key+"_sha256"]=sha256_path(item[key])
manifest_path=ARTIFACTS/"manifest_benchmark.json"
manifest_path.write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding="utf-8")
if IS_KAGGLE:
    PERSIST_ROOT.mkdir(parents=True,exist_ok=True)
    for p in ARTIFACTS.iterdir(): shutil.copy2(p,PERSIST_ROOT/p.name)
    print("Tệp đã lưu để tải từ Kaggle:",PERSIST_ROOT)
else:
    archive=shutil.make_archive(str(ARTIFACTS),"zip",root_dir=ARTIFACTS)
    print("Gói MP4 và manifest:",archive)
from IPython.display import display,Video
for item in results:
    print(item["name"],item["probe_normalized"])
    display(Video(filename=item["normalized"],embed=False,width=270))


## 9. Đóng ComfyUI

Chạy cell này sau khi đã lưu và xem xong kết quả. Nếu notebook bị ngắt trước đó, runtime sẽ tự kết thúc theo vòng đời của nền tảng.

In [ ]:
if "server" in globals() and server.poll() is None:
    server.terminate()
    try: server.wait(timeout=20)
    except subprocess.TimeoutExpired: server.kill()
print("Đã dừng ComfyUI nội bộ.")
